# Sample characteristics (Table 1) and supplementary descriptive tables

Produces Table 1, the sample-selection cascade (Supplementary Table S1), the
per-variable missingness table (Supplementary Table S2) and the full
distributional summary behind Table 1 (Supplementary Table S13). All weighted
statistics use the four-year MEC examination weight (WTMEC4YR = WTMEC2YR / 2)
for two pooled two-year cycles. Figure 1 is produced by `scripts/14_main_fig1.ipynb`.

## Weighted statistics

Let $x_i$ be the value for participant $i$ and $w_i$ the survey weight
(WTMEC4YR). Weighted mean and weighted (uncorrected) variance:

$$\hat{\mu}_w = \frac{\sum_i w_i x_i}{\sum_i w_i}, \quad \hat{\sigma}^2_w = \frac{\sum_i w_i (x_i - \hat{\mu}_w)^2}{\sum_i w_i}$$

The standard error of the weighted mean uses the Kish effective sample size
(Kish 1965, *Survey Sampling*):

$$n_{\text{eff}} = \frac{\left(\sum_i w_i\right)^2}{\sum_i w_i^2}, \quad \mathrm{SE}(\hat{\mu}_w) = \sqrt{\frac{\hat{\sigma}^2_w}{n_{\text{eff}}}}$$

Weighted median and quartiles are read off the weighted empirical CDF
$F_w(t) = \left(\sum_i w_i \mathbb{1}\{x_i \le t\}\right) / \sum_i w_i$:

$$\hat{q}_p = \inf\{t : F_w(t) \ge p\}, \quad p \in \{0.25, 0.50, 0.75\}$$

For a categorical value $c$ with indicator $y_i = \mathbb{1}\{x_i = c\}$, the
weighted proportion and its standard error are:

$$\hat{p}_w = \frac{\sum_i w_i y_i}{\sum_i w_i}, \quad \mathrm{SE}(\hat{p}_w) = \sqrt{\frac{\hat{p}_w (1 - \hat{p}_w)}{n_{\text{eff}}}}$$

`across_strata_p` gives a one-way ANOVA F-test (continuous) or Pearson
chi-square test (categorical) across the four age strata
(18-39 / 40-59 / 60-69 / 70+); Table 1 itself carries no p-value column.

## Derived variables

$$\mathrm{PHQ9\_total} = \sum_{k=1}^{9} \mathrm{DPQ}_k, \quad \text{undefined if any } \mathrm{DPQ}_k \text{ missing}$$

$$\mathrm{PHQ9\_clinical} = \mathbb{1}\{\mathrm{PHQ9\_total} \ge 10\}$$

Cognitive composite (within the 60+ subsample):

$$\mathrm{Memory\_z} = \frac{\mathrm{memory\_score} - \bar{\mathrm{memory\_score}}_{60+}}{\mathrm{SD}(\mathrm{memory\_score})_{60+}}, \quad \mathrm{Fluency\_z}, \mathrm{Speed\_z} \text{ defined analogously}$$

$$\mathrm{Cog\_composite\_z} = \frac{\mathrm{Memory\_z} + \mathrm{Fluency\_z} + \mathrm{Speed\_z}}{3}$$

Raw domain scores of zero are recoded to missing before z-standardisation (a
zero marks a non-administered sub-test). The same definition is used in
`scripts/9_mplus_prep.ipynb`.

## Inputs

- `data/processed/8_nhanes_analytic.csv` — participant-level analytic dataset.

## Outputs

- `result/main/Table1_Sample_Characteristics.csv` and `.xlsx` (Excel with a Notes sheet).
- `result/supplementary/SuppTableS13_Distribution_Summary.csv` and `.xlsx` — median, IQR, mean, SD, SE and skewness for every continuous characteristic.
- `result/supplementary/SuppTableS1_Sample_Cascade.csv` and `.xlsx` — sample-selection cascade counts.
- `result/supplementary/SuppTableS2_Missingness.csv` and `.xlsx` — per-variable missingness in adults.

## Samples

Table 1 describes the structural-equation-model samples: adults aged 18 years
and over with non-missing exposure (IV), weight and core covariates (age, sex,
body mass index, education, accelerometer wear time, self-reported sleep
duration) and at least one observed dependent variable (KDM_BA_advance,
SII_log, PHQ9_total), shown overall and split at age 60. The cognition sample
is the 60+ subset with at least one of KDM_BA_advance, SII_log, Memory_z,
Fluency_z, Speed_z observed. Supplementary Table S1 also reports the narrower
complete-case samples.


## Step 0 — Imports and paths


In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches
warnings.filterwarnings('ignore')

import os as _os

def find_root():
    """Locate the project root without assuming where the kernel started."""
    env = _os.environ.get('NHANES_SEM_ROOT')
    cands = ([Path(env)] if env else []) + [Path.cwd(), *Path.cwd().parents]
    for q in cands:
        if (q / 'data' / 'processed').is_dir() and (q / 'scripts').is_dir():
            return q
    raise RuntimeError(
        'cannot find the project root from ' + str(Path.cwd())
        + ' -- start Jupyter inside the project, or set NHANES_SEM_ROOT')

BASE = find_root()
PROC   = BASE / 'data' / 'processed'
R_MAIN = BASE / 'result' / 'main'
R_SUPP = BASE / 'result' / 'supplementary'
R_RAW  = BASE / 'result' / 'raw'
F_MAIN = BASE / 'figure' / 'main'
F_SUPP = BASE / 'figure' / 'supplementary'
F_SRC  = BASE / 'figure' / 'source_data'
for p in (R_MAIN, R_SUPP, R_RAW, F_MAIN, F_SUPP, F_SRC):
    p.mkdir(parents=True, exist_ok=True)

# Shared colour palette (see scripts/nature_palette.py)
sys.path.insert(0, str(BASE / 'scripts'))

# Reload project modules so the notebook uses the code on disk.
import importlib
import wls_utils, nature_palette
for _m in (wls_utils, nature_palette,):
    importlib.reload(_m)

from nature_palette import (IV_GRADIENT_4, KDM_BA_COLOR, SII_COLOR,
                            PHQ9_COLOR, COG_COLOR, GREY, WONG)
from wls_utils import chi2_sf, f_sf


## Step 1 — Load and derive variables

Reads the analytic file, derives PHQ9_total, PHQ9_clinical and the within-60+
cognition z-scores, checks WTMEC4YR, and adds display names, race/education
categories and the four age strata.


In [ ]:
df = pd.read_csv(PROC / '8_nhanes_analytic.csv')
print(f'Loaded: {df.shape}')

# PHQ-9 total (refusal=7 and dont-know=9 set to missing)
dpq_cols = [f'DPQ{i:03d}' for i in [10,20,30,40,50,60,70,80,90]]
phq = df[dpq_cols].replace({7: np.nan, 9: np.nan})
df['PHQ9_total'] = phq.sum(axis=1, min_count=9)
df['PHQ9_clinical'] = (df['PHQ9_total'] >= 10).astype('Int64')

# Cognition composite (60+): raw zeros -> missing, then z-standardised within
# the 60+ subsample; same definition as scripts/9_mplus_prep.ipynb.
elig_60 = df['RIDAGEYR'] >= 60
for src_col, out_col in [('memory_score',  'Memory_z'),
                         ('fluency_score', 'Fluency_z'),
                         ('speed_score',   'Speed_z')]:
    raw = df[src_col].replace(0, np.nan)
    s   = raw.loc[elig_60].dropna()
    df[out_col] = np.where(elig_60, (raw - s.mean()) / s.std(), np.nan)
df['Cog_composite_z'] = df[['Memory_z', 'Fluency_z', 'Speed_z']].mean(axis=1, skipna=False)

# WTMEC4YR is stored in the analytic file; checked here, not re-derived.
assert 'WTMEC4YR' in df.columns, 'WTMEC4YR missing from the analytic file'
assert np.allclose(df['WTMEC4YR'], df['WTMEC2YR'] / 2.0, equal_nan=True), \
    'stored WTMEC4YR does not match WTMEC2YR / 2 -- the file is stale'
df['Age']         = df['RIDAGEYR']
df['Sex']         = df['RIAGENDR']
df['BMI']         = df['BMXBMI']
df['Edu']         = df['DMDEDUC2']
df['Sleep_hours'] = df['SLD010H']
df['PIR']         = df['INDFMPIR']

race_map = {1: 'Mexican American', 2: 'Other Hispanic', 3: 'Non-Hispanic White',
            4: 'Non-Hispanic Black', 6: 'Non-Hispanic Asian', 7: 'Other/Multiracial'}
df['Race'] = df['RIDRETH3'].map(race_map)
edu_map = {1: '<HS', 2: '<HS', 3: 'HS/GED', 4: 'Some college', 5: 'College graduate'}
df['Edu_cat'] = df['Edu'].map(edu_map)

def age_stratum(age):
    if pd.isna(age) or age < 18: return None
    if age < 40: return '18-39'
    if age < 60: return '40-59'
    if age < 70: return '60-69'
    return '70+'
df['Age_stratum'] = df['Age'].apply(age_stratum)
strata = ['18-39', '40-59', '60-69', '70+']

## Step 2 — Define analytic samples

Builds the depression and cognition model samples (complete exposure, weight
and covariates; at least one observed dependent variable) and the complete-case
subsets used in Supplementary Table S1.


In [ ]:
COVS = ['Age', 'Sex', 'BMI', 'Edu', 'Wear_Time', 'Sleep_hours']
adults = df[df['Age'] >= 18].copy()
elder  = df[df['Age'] >= 60].copy()

# Table 1 describes the model samples: complete x-variables and at least one
# observed dependent variable, as Mplus retains under FIML.
x_complete = adults.dropna(subset=['IV', 'WTMEC4YR'] + COVS).copy()
_anyY = x_complete[['KDM_BA_advance', 'SII_log', 'PHQ9_total']].notna().any(axis=1)
SEM_DEP = x_complete[_anyY].copy()
SEM_DEP_YOUNG = SEM_DEP[SEM_DEP['Age'] < 60]
SEM_DEP_OLD   = SEM_DEP[SEM_DEP['Age'] >= 60]

_x60 = x_complete[x_complete['Age'] >= 60]
_cogY = ['KDM_BA_advance', 'SII_log', 'Memory_z', 'Fluency_z', 'Speed_z']
SEM_COG = _x60[_x60[[c for c in _cogY if c in _x60.columns]].notna().any(axis=1)].copy()

# One column per analysis sample, with n in the column header. The cognition
# sample (n = 2,908) differs from the 60+ depression sample (n = 2,909) by one
# participant and agrees on every statistic to two decimals, so it is not
# shown as a separate column; the difference is footnoted.
T1_COLS = [(f'Overall (n = {len(SEM_DEP):,})', SEM_DEP),
           (f'Aged 18\u201359 years (n = {len(SEM_DEP_YOUNG):,})', SEM_DEP_YOUNG),
           (f'Aged \u226560 years (n = {len(SEM_DEP_OLD):,})', SEM_DEP_OLD)]

# Complete-case subsets, reported in Supplementary Table S1.
adults_dep_u = adults.dropna(subset=['IV', 'KDM_BA', 'SII', 'PHQ9_total'] + COVS).copy()
elder_cog_u  = elder.dropna(subset=['IV', 'KDM_BA', 'SII', 'Cog_composite_z'] + COVS).copy()
T1_SAMPLE = SEM_DEP

for lab, s in T1_COLS:
    print(f'{lab:20s} n = {len(s):,}')
print()
print(f'complete-case subsets (Supp Table S1): depression {len(adults_dep_u):,}, '
      f'cognition {len(elder_cog_u):,}')


## Step 3 — Weighted summary helpers

Weighted mean/SE, weighted proportion/SE, weighted median/IQR and the
across-strata test, as defined in the header.


In [ ]:
def weighted_mean_se(values, weights):
    """Weighted mean and its SE from the Kish effective sample size n_eff = (sum w)^2 / sum w^2."""
    mask = ~(np.isnan(values) | np.isnan(weights))
    if mask.sum() < 5: return np.nan, np.nan
    v, w = values[mask], weights[mask]
    mean = np.average(v, weights=w)
    var = np.average((v - mean) ** 2, weights=w)
    n_eff = w.sum() ** 2 / (w ** 2).sum()
    return mean, float(np.sqrt(var / n_eff))

def weighted_pct(values, weights, target):
    """Weighted percentage of values == target, with SE = sqrt(p(1-p)/n_eff) * 100."""
    mask = ~(np.isnan(values) | np.isnan(weights))
    if mask.sum() < 5: return np.nan, np.nan
    v, w = values[mask], weights[mask]
    p = np.average((v == target), weights=w)
    n_eff = w.sum() ** 2 / (w ** 2).sum()
    return p * 100, float(np.sqrt(p * (1 - p) / n_eff) * 100)

def weighted_median_iqr(values, weights):
    """Weighted median, Q1 and Q3 read off the weighted empirical CDF."""
    mask = ~(np.isnan(values) | np.isnan(weights))
    if mask.sum() < 5: return np.nan, np.nan, np.nan
    v, w = values[mask].to_numpy(), weights[mask].to_numpy()
    order = np.argsort(v)
    vs, ws = v[order], w[order]
    cw = np.cumsum(ws); total = cw[-1]
    return (vs[np.searchsorted(cw, 0.50 * total)],
            vs[np.searchsorted(cw, 0.25 * total)],
            vs[np.searchsorted(cw, 0.75 * total)])

def across_strata_p(values, group, kind='continuous'):
    """p-value of a one-way ANOVA F (continuous) or Pearson chi-square (categorical) test across strata, via wls_utils.f_sf / chi2_sf."""
    mask = ~values.isna() & ~group.isna()
    v, g = values[mask], group[mask]
    if kind == 'continuous':
        arrays = [v[g == s].to_numpy() for s in g.unique() if (g == s).sum() >= 3]
        if len(arrays) < 2:
            return np.nan
        k = len(arrays)
        N = sum(len(a) for a in arrays)
        gm = np.concatenate(arrays).mean()
        ssb = sum(len(a) * (a.mean() - gm) ** 2 for a in arrays)
        ssw = sum(((a - a.mean()) ** 2).sum() for a in arrays)
        if ssw <= 0 or N <= k:
            return np.nan
        F = (ssb / (k - 1)) / (ssw / (N - k))
        return f_sf(F, k - 1, N - k)
    ct = pd.crosstab(v, g).to_numpy(dtype=float)
    if ct.size == 0 or ct.sum() == 0:
        return np.nan
    row = ct.sum(1, keepdims=True); col = ct.sum(0, keepdims=True)
    E = row @ col / ct.sum()
    with np.errstate(divide='ignore', invalid='ignore'):
        chi2 = np.nansum(np.where(E > 0, (ct - E) ** 2 / E, 0.0))
    dof = (ct.shape[0] - 1) * (ct.shape[1] - 1)
    return chi2_sf(chi2, dof) if dof > 0 else np.nan

def fmt_mean_se(m, se, dp=2):
    return '—' if pd.isna(m) else f'{m:.{dp}f} ({se:.{dp}f})'

def fmt_median_iqr(med, q25, q75, dp=1):
    return '—' if pd.isna(med) else f'{med:.{dp}f} ({q25:.{dp}f}–{q75:.{dp}f})'

def fmt_p(p):
    if pd.isna(p): return '—'
    return '<0.001' if p < 0.001 else f'{p:.3f}'

## Step 4 — Construct Table 1

Continuous rows: survey-weighted median (IQR); categorical rows: unweighted n
with survey-weighted %. A cell is left blank when fewer than 30 participants
in the column have the characteristic observed.


In [ ]:
# Continuous variables: median (IQR); categorical: n (%) (Shim et al. 2026,
# npj Aging). Rows are grouped in labelled blocks and every row label carries
# its summary statistic. No p-value column: the columns are overlapping
# samples.
BLOCKS = [
    ('Demographic and clinical characteristics', [
        ('med', 'Age, years',                          'Age'),
        ('bin', 'Female sex',                          'Sex', 1),
        ('cat', 'Race and ethnicity',                  'Race',
         ['Non-Hispanic White', 'Non-Hispanic Black', 'Mexican American',
          'Other Hispanic', 'Non-Hispanic Asian', 'Other/Multiracial']),
        ('cat', 'Educational attainment',              'Edu_cat',
         ['<HS', 'HS/GED', 'Some college', 'College graduate']),
        ('med', 'Family income-to-poverty ratio',      'PIR'),
        ('med', 'Body mass index, kg/m\u00b2',            'BMI'),
        ('med', 'Sleep duration, h',                   'Sleep_hours'),
        ('med', 'Accelerometer wear time, min/day',    'Wear_Time'),
    ]),
    ('Rest\u2013activity rhythm metrics', [
        ('med', 'Relative amplitude',                  'RA'),
        ('med', 'Interdaily stability',                'IS'),
        ('med', 'Intradaily variability',              'IV'),
    ]),
    # Total activity volume is the only activity quantity that enters a model
    # (volume-adjusted specifications). Peak 30-min MIMS, minutes above the
    # intensity threshold and sedentary time enter no model and are described
    # in Supplementary Table S13.
    ('Physical activity characteristics', [
        ('med', 'Total daily activity volume, MIMS',   'TOTAL_MIMS'),
    ]),
    # Block label matches the Methods heading (cross-sectional design).
    ('Candidate biological measures', [
        ('med', 'KDM biological age, years',           'KDM_BA'),
        ('med', 'KDM biological-age advancement, years', 'KDM_BA_advance'),
        ('med', 'Systemic immune-inflammation index, \u00d710\u2079/L', 'SII'),
    ]),
    ('Health outcomes', [
        ('med', 'PHQ-9 depressive-symptom score',      'PHQ9_total'),
        ('bin', 'PHQ-9 score \u2265 10',                  'PHQ9_clinical', 1),
        ('med', 'Cognitive function composite, z score', 'Cog_composite_z'),
    ]),
]
STAT_LABEL = {'med': 'median (IQR)', 'bin': 'n (%)', 'cat': 'n (%)'}

def cell_median(sub, col):
    v = sub[col].dropna()
    if len(v) < 30:
        return '\u2014'
    med, q25, q75 = weighted_median_iqr(sub[col], sub['WTMEC4YR'])
    dp = 2 if abs(med) < 10 else (1 if abs(med) < 1000 else 0)
    # Unicode minus sign; ' to ' instead of an en-dash when a bound is negative.
    def f(x):
        return f'{x:,.{dp}f}'.replace('-', '\u2212')
    sep = ' to ' if (q25 < 0 or q75 < 0) else '\u2013'
    return f'{f(med)} ({f(q25)}{sep}{f(q75)})'

def cell_count_pct(sub, col, target):
    v = sub[col].dropna()
    if not len(v):
        return '\u2014'
    n = int((v == target).sum())
    pct, _ = weighted_pct(sub[col], sub['WTMEC4YR'], target)
    return f'{n:,} ({pct:.1f})'

rows = []

for block_name, items in BLOCKS:
    rows.append({'Characteristic': block_name, **{lab: '' for lab, _ in T1_COLS}})
    for item in items:
        kind, label, col = item[0], item[1], item[2]
        pretty = f'  {label}, {STAT_LABEL[kind]}'
        if kind == 'med':
            rows.append({'Characteristic': pretty,
                         **{lab: cell_median(s, col) for lab, s in T1_COLS}})
        elif kind == 'bin':
            rows.append({'Characteristic': pretty,
                         **{lab: cell_count_pct(s, col, item[3]) for lab, s in T1_COLS}})
        else:
            rows.append({'Characteristic': pretty,
                         **{lab: '' for lab, _ in T1_COLS}})
            for lvl in item[3]:
                rows.append({'Characteristic': f'    {lvl}',
                             **{lab: cell_count_pct(
                                 s.assign(_m=(s[col] == lvl).astype(int)), '_m', 1)
                                for lab, s in T1_COLS}})

table1 = pd.DataFrame(rows)
print(table1.to_string(index=False))


## Step 5 — Save Table 1 and Supplementary Table S13

Writes Table 1 (CSV and Excel with a Notes sheet) and Supplementary Table S13,
the full distributional summary (median, IQR, mean, SD, SE, skewness) of every
continuous characteristic plus four characteristics that enter no model.


In [ ]:
(R_MAIN / 'Table1_Sample_Characteristics.csv').write_text(table1.to_csv(index=False))

_kdm_mean = np.average(SEM_DEP['KDM_BA_advance'].dropna(),
                       weights=SEM_DEP.loc[SEM_DEP['KDM_BA_advance'].notna(), 'WTMEC4YR'])
NOTES = [
    'Continuous variables are reported as survey-weighted medians '
    '(interquartile ranges) because several showed skewed distributions; '
    'categorical variables are reported as unweighted counts with '
    'survey-weighted percentages. Means, standard deviations and skewness for '
    'every characteristic and column are given in Supplementary Table S13.',
    'Estimates incorporate the four-year NHANES mobile examination-centre '
    'weight and account for the complex survey design.',
    f'The overall analytic sample included {len(SEM_DEP):,} adults, comprising '
    f'{len(SEM_DEP_YOUNG):,} adults aged 18-59 years and {len(SEM_DEP_OLD):,} '
    f'adults aged 60 years and over. The cognitive-function analysis included '
    f'{len(SEM_COG):,} adults aged 60 years and over.',
    'KDM biological-age advancement was defined as KDM biological age minus '
    'chronological age, with positive values indicating more advanced '
    'biological age relative to chronological age.',
    'Cognitive testing was administered only to adults aged 60 years and over. '
    'The cognitive composite shown here is provided for descriptive purposes; '
    'cognitive function was modelled as a latent factor in the primary '
    'structural models.',
    'Abbreviations: IQR, interquartile range; KDM, Klemera-Doubal method; '
    'MIMS, Monitor-Independent Movement Summary; PHQ-9, 9-item Patient Health '
    'Questionnaire.',
]


with pd.ExcelWriter(R_MAIN / 'Table1_Sample_Characteristics.xlsx', engine='openpyxl') as xw:
    table1.to_excel(xw, sheet_name='Table 1', index=False)
    pd.DataFrame({'Notes': NOTES}).to_excel(xw, sheet_name='Notes', index=False)
print(f'wrote {R_MAIN / "Table1_Sample_Characteristics.xlsx"}')

# Supplementary Table S13: median, IQR, mean, SD, SE of the mean and skewness
# for every continuous Table 1 characteristic, plus four characteristics that
# enter no model (Mplus specifications or supportive regressions): peak 30-min
# MIMS, minutes above the MIMS intensity threshold, sedentary time and NLR.
# 'Minutes above the MIMS intensity threshold' is not labelled
# moderate-to-vigorous activity: the threshold maps Montoye et al. (2020)
# non-dominant-wrist activity-count cut-points to MIMS via Karas et al. (2022),
# and no consensus MIMS intensity cut-point exists.
SUPP_ONLY = [
    ('Peak 30-min activity, MIMS',                     'peak30_mims'),
    ('Minutes above the MIMS intensity threshold, min/day', 'obj_mvpa_min'),
    ('Sedentary time, min/day',                        'obj_sed_min'),
    ('Neutrophil-to-lymphocyte ratio',                 'NLR'),
]

raw_rows = []
for label, col in ([(i[1], i[2]) for _, items in BLOCKS for i in items
                    if i[0] == 'med'] + SUPP_ONLY):
    for lab, sub in T1_COLS:
        v = sub[col].dropna()
        if len(v) < 30:
            continue
        med, q25, q75 = weighted_median_iqr(sub[col], sub['WTMEC4YR'])
        m, se = weighted_mean_se(sub[col], sub['WTMEC4YR'])
        _m = sub[col].notna()
        _v, _w = sub.loc[_m, col].values, sub.loc[_m, 'WTMEC4YR'].values
        _mu = np.average(_v, weights=_w)
        _sd = float(np.sqrt(np.average((_v - _mu) ** 2, weights=_w)))
        _skew = float(np.average(((_v - _mu) / _sd) ** 3, weights=_w)) if _sd > 0 else np.nan
        # Decimal places follow the SD, not the level.
        _dp = 0 if _sd >= 100 else (1 if _sd >= 10 else (2 if _sd >= 1 else 3))
        # SE of the mean to three significant figures.
        _sedp = max(0, 2 - int(np.floor(np.log10(abs(se))))) if se > 0 else 3
        raw_rows.append({
            'Characteristic': label,
            'Column': lab,
            'n observed': f'{int(v.shape[0]):,}',
            'Median': f'{med:,.{_dp}f}',
            'Q1': f'{q25:,.{_dp}f}',
            'Q3': f'{q75:,.{_dp}f}',
            'Mean': f'{m:,.{_dp}f}',
            'SD': f'{_sd:,.{_dp}f}',
            'SE of mean': f'{se:,.{_sedp}f}',
            'Skewness': f'{_skew:.2f}',
        })
tableS13 = pd.DataFrame(raw_rows)

S13_NOTES = [
    'Characteristics listed after the cognitive composite -- peak 30-min '
    'activity, minutes above the MIMS intensity threshold, sedentary time and '
    'the neutrophil-to-lymphocyte ratio -- are described here rather than in '
    'Table 1 because none of them enters any model, primary, secondary or '
    'sensitivity. Minutes above the MIMS intensity threshold are not labelled '
    'moderate-to-vigorous activity: the underlying activity-count-to-MIMS '
    'mapping is from wrist-worn calibration data matching the NHANES protocol, '
    'but no consensus MIMS intensity cut-point exists, so the variable is '
    'named for what was computed.',
    'Full distributional summary of every continuous characteristic in '
    'Table 1, within each analysis column. All estimates are survey-weighted '
    'using the four-year mobile examination centre weight.',
    'Table 1 reports the median and interquartile range. This table gives the '
    'skewness that motivates that choice: nine of the eighteen '
    'characteristics have absolute skewness above 1, including relative '
    'amplitude, intradaily variability, KDM biological-age advancement, the '
    'systemic immune-inflammation index, the neutrophil-to-lymphocyte ratio '
    'and the PHQ-9. For these, the mean is displaced from the typical value '
    '(for example the PHQ-9 mean is 3.1 against a median of 1).',
    'The standard deviation is included because effect sizes in the Results '
    'are expressed per standard deviation of exposure and of outcome; this '
    'table is where a reader converts those coefficients into natural units.',
    'Numeric precision follows the standard deviation of each characteristic '
    'rather than its level. The SE of the mean is the Kish '
    'effective-sample-size standard error, a precision measure and not a '
    'measure of spread; it is given to three significant figures.',
    'Characteristics observed in fewer than 30 participants within a column '
    'are omitted from that column; the cognition composite is measured only '
    'at 60 years and over.',
]
tableS13.to_csv(R_SUPP / 'SuppTableS13_Distribution_Summary.csv', index=False)
with pd.ExcelWriter(R_SUPP / 'SuppTableS13_Distribution_Summary.xlsx',
                    engine='openpyxl') as xw:
    tableS13.to_excel(xw, sheet_name='Supp Table S13', index=False)
    pd.DataFrame({'Notes': S13_NOTES}).to_excel(xw, sheet_name='Notes', index=False)
print(f'wrote {R_SUPP / "SuppTableS13_Distribution_Summary.xlsx"} '
      f'({len(tableS13)} rows)')

# Move result/raw/Table1_unrounded.csv, if present, to _superseded/.
_old = R_RAW / 'Table1_unrounded.csv'
if _old.exists():
    _dest = BASE / '_superseded' / _old.name
    _dest.parent.mkdir(exist_ok=True)
    _old.rename(_dest)
    print(f'superseded -> _superseded/{_old.name}')


## Step 6 — Supplementary Tables S1 (cascade) and S2 (missingness)

Unweighted counts at each selection step, and per-variable missingness among
all adults aged 18 years and over. Both are written as CSV and as Excel with a
Notes sheet.


In [ ]:
# Supplementary Table S1: sample-selection cascade, unweighted counts.
cascade = pd.DataFrame([
    ('Pooled NHANES 2011\u20132014 sample',                          len(df)),
    ('Adults aged \u226518 years',                                   len(adults)),
    ('Adults with valid rest\u2013activity rhythm metrics',           int(adults['IV'].notna().sum())),
    ('Adults with both biological measures observed',                int(adults.dropna(subset=['IV','KDM_BA','SII']).shape[0])),
    ('Adults with complete PHQ-9',                                   int(adults.dropna(subset=['IV','KDM_BA','SII','PHQ9_total']).shape[0])),
    ('Complete-case depression sample (all covariates observed)',    len(adults_dep_u)),
    ('Adults aged \u226560 years',                                   len(elder)),
    ('Complete-case cognition sample (all covariates observed)',     len(elder_cog_u)),
], columns=['cascade_step', 'n'])
cascade.columns = ['Selection step', 'n']
cascade.to_csv(R_SUPP / 'SuppTableS1_Sample_Cascade.csv', index=False)
with pd.ExcelWriter(R_SUPP / 'SuppTableS1_Sample_Cascade.xlsx',
                    engine='openpyxl') as xw:
    cascade.to_excel(xw, sheet_name='Supp Table S1', index=False)
    pd.DataFrame({'Notes': [
        'Unweighted counts at each selection step, from the pooled NHANES '
        '2011-2014 file to the analytic universes.',
        'The complete-case universes shown here are narrower than the '
        'structural equation model samples, which retain participants with '
        'complete exogenous variables and at least one observed dependent '
        'variable. Supplementary Table S12 gives that accounting in full.',
    ]}).to_excel(xw, sheet_name='Notes', index=False)
print(cascade.to_string(index=False))

miss_vars = ['IV','RA','IS','KDM_BA','KDM_BA_advance','SII','NLR','MLR',
             'PHQ9_total','Cog_composite_z','obj_mvpa_min','obj_sed_min',
              'peak30_mims','TOTAL_MIMS',
             'Age','Sex','BMI','Edu','Race','PIR','Wear_Time','Sleep_hours']
# Display names for the missingness table (same names as Supplementary Table S13).
MISS_LABEL = {
    'IV': 'Intradaily variability (IV)', 'RA': 'Relative amplitude (RA)',
    'IS': 'Interdaily stability (IS)',
    'KDM_BA': 'KDM biological age', 'KDM_BA_advance': 'KDM biological-age advancement',
    'SII': 'Systemic immune-inflammation index (SII)',
    'NLR': 'Neutrophil-to-lymphocyte ratio', 'MLR': 'Monocyte-to-lymphocyte ratio',
    'PHQ9_total': 'PHQ-9 total score', 'Cog_composite_z': 'Cognitive composite (z score)',
    'obj_mvpa_min': 'Minutes above the MIMS intensity threshold',
    'obj_sed_min': 'Sedentary time', 'peak30_mims': 'Peak 30-min activity',
    'TOTAL_MIMS': 'Total daily activity volume',
    'Age': 'Age', 'Sex': 'Sex', 'BMI': 'Body mass index', 'Edu': 'Education',
    'Race': 'Race and ethnicity', 'PIR': 'Family income-to-poverty ratio',
    'Wear_Time': 'Accelerometer wear time', 'Sleep_hours': 'Sleep duration',
}
miss_rows = []
for v in miss_vars:
    if v not in adults.columns: continue
    n_present = int(adults[v].notna().sum())
    miss_rows.append({'variable': MISS_LABEL.get(v, v), 'n_present_adults': n_present,
                      'n_missing_adults': int(len(adults) - n_present),
                      'pct_missing_adults': round((1 - n_present / len(adults)) * 100, 2)})
miss = pd.DataFrame(miss_rows).rename(columns={
    'variable': 'Variable', 'n_present_adults': 'n observed',
    'n_missing_adults': 'n missing', 'pct_missing_adults': '% missing'})
miss.to_csv(R_SUPP / 'SuppTableS2_Missingness.csv', index=False)
with pd.ExcelWriter(R_SUPP / 'SuppTableS2_Missingness.xlsx',
                    engine='openpyxl') as xw:
    miss.to_excel(xw, sheet_name='Supp Table S2', index=False)
    pd.DataFrame({'Notes': [
        'Item-level missingness among all adults aged 18 years and over '
        f'(n = {len(adults):,}), before any analytic restriction.',
        'Counts are unweighted. Education is the largest single source of '
        'exclusion from the models; Supplementary Table S12 shows the loss '
        'covariate by covariate.',
    ]}).to_excel(xw, sheet_name='Notes', index=False)
print(f'wrote {R_SUPP / "SuppTableS2_Missingness.xlsx"}')